# EXP-2 — Hybrid Retrieval (BM25 + Dense)

BM25 handles exact keyword matching. Dense retrieval handles semantic similarity. EnsembleRetriever combines both with reciprocal rank fusion. Covers the cases each one misses on its own.

## Setup

In [25]:
import os
import sys
import time
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings("ignore")

print("all imports loaded")


all imports loaded


In [5]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [6]:

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [7]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("HR_RAG_Experiments")

print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")


mlflow tracking uri: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow


## Load and Prepare Data

In [8]:
# adjust the path and glob pattern to match your folder structure
loader = DirectoryLoader("../data/policies", glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [10]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5108.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [11]:
print(f"eval set: {len(dataset['question'])} question-answer pairs")


eval set: 10 question-answer pairs


In [13]:
from langchain_groq import ChatGroq

ragas_llm_g = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [14]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [15]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)


In [16]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(ragas_llm_g, run_config=RunConfig(max_workers=1))

In [ ]:
import nest_asyncio
nest_asyncio.apply()

def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            answer = chain.invoke(question)
            docs = get_docs_fn(question)
            answers.append(answer)
            contexts.append([d.page_content for d in docs])
        except Exception as e:
            print(f"  error on: {question[:60]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    })

    # max_workers=1 means sequential — no parallel calls, no rate limit hits
    # timeout=120 because Groq free tier can be slow sometimes
    run_cfg = RunConfig(max_workers=1, timeout=120)

    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,   # if one question fails,we still want to get metrics on the rest of the eval set
    )
    return result

In [18]:
def measure_latency(chain, test_q="What is the leave policy?"):
    start = time.time()
    chain.invoke(test_q)
    return round(time.time() - start, 3)


In [ ]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
           
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment-Specific Imports

In [20]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever

## Experiment

In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

dense_retriever = db.as_retriever(search_kwargs={"k": 3})

# equal weight to both strategies, we can tune this
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ragas_llm_g
    | StrOutputParser()
)

print(chain.invoke("What is the leave policy?"))


The leave policy includes the following: 

- Earned Leave / Privilege Leave (PL/EL) with entitlement of 18 days per calendar year, accrual of 1.5 days per month after 6 months of service, and a carry forward of up to 30 days per year with a maximum accumulation of 90 days.
- Leave Without Pay (LWP) applicable when all leave balance is exhausted and requires explicit approval from Department Head and HR.
- Work From Home (WFH) Leave entitlement of up to 3 days per week for eligible roles, subject to manager approval.
- Maternity leave with full pay during the leave period (100% of last drawn salary) and creche facility entitlement for employees returning from maternity leave.


In [24]:
result = evaluate_rag(chain, ensemble_retriever.invoke, dataset)
latency = measure_latency(chain)

Evaluating: 100%|██████████| 30/30 [16:54<00:00, 33.82s/it]


In [27]:
result,latency

({'faithfulness': 0.8444, 'context_precision': 0.5583, 'context_recall': 0.7133},
 4.288)

In [30]:
log_to_mlflow(
    run_name="EXP-2-HYBRID",
    result=result,
    latency=latency,
    retriever_type="hybrid-bm25-dense",
    top_k=3,
    extra_params={"bm25_weight": 0.5, "dense_weight": 0.5},
)


🏃 View run EXP-2-HYBRID at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1/runs/3853e096a64e48b790d1288be123040a
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1
